In [6]:
%matplotlib inline
import torch
from d2l import torch as d2l

torch.set_printoptions(2)  # 精简输出精度
def box_iou(boxes1,boxes2):
    """计算两个锚框或边界框列表中承兑的交并比"""
    box_area = lambda boxes:((boxes[:,2]-boxes[:,0])*(boxes[:,3]-boxes[:,1]))
#     定义函数：输入两组框
# boxes1 形状 (N, 4)
# boxes2 形状 (M, 4)
# 每行是一个框坐标 (xmin, ymin, xmax, ymax)。
#lambda 参数1, 参数2, ... : 表达式
    
    #boxes1,boxes2,areas1和areas2的形状分别为
    #(boxes1的数量,4)
    #(boxes2的数量,4)
    #(boxes1的数量，）
    #(boxes2的数量，）
    areas1 = box_area(boxes1)
    areas2 = box_area(boxes2)
    #inter_upperlefts、inter_lowerrights、inters的形状为
    #(boxes1的数量，boxes2的数量,2)
    inter_upperlefts = torch.max(boxes1[:,None,:2],boxes2[:,:2])
    inter_lowerrights = torch.min(boxes1[:,None,2:],boxes2[:,2:])
    
#     它们的形状都是：
# inter_upperlefts.shape == (N, M, 2)
# inter_lowerrights.shape == (N, M, 2)
# 第三维 2 代表 (x, y)。
    
    inters = (inter_lowerrights-inter_upperlefts).clamp(min=0)
    # clamp(min=0) 表示：小于 0 的值全部变成 0，大于等于 0 的保持不变。
    #inter_areasandunion_areas的形状为(Boxes1的数量,boxes2的数量)
    inter_areas = inters[:,:,0]*inters[:,:,1]
#     inters[:,:,0] 是宽 w，形状 (N,M)
# inters[:,:,1] 是高 h，形状 (N,M)
# 相乘得到 (N,M) 的交集面积矩阵
    
    union_areas = areas1[:,None]+areas2 - inter_areas
#     areas1[:,None] 把 (N,) 变 (N,1)，广播到 (N,M)
# areas2 是 (M,)，广播到 (N,M)
# 减去 inter_areas 得到 (N,M)
    
    return inter_areas/union_areas

In [7]:
#13.4.3在训练数据中标注锚框
def assign_anchor_to_bbox(ground_truth,anchors,device,iou_threshold=0.5):
    """将最接近的真实边界框分配给锚框"""
    num_anchors,num_gt_bboxes = anchors.shape[0],ground_truth.shape[0]
    #位于第i行和第j列的元素x_ij是锚框i和真实边界框j的IoU
    jaccard = box_iou(anchors,ground_truth)
    #对于每个锚框,分配的真实边界框的张量
    anchors_bbox_map = torch.full((num_anchors,),-1,dtype=torch.long,device=device)
    
#     torch.full((num_anchors,), -1, ...)
# torch.full(shape, fill_value) 会生成一个指定形状的张量，并把所有元素初始化为同一个值。
# 这里的 shape 是 (num_anchors,)：表示一维向量，长度是 num_anchors。
# fill_value 是 -1：所有位置先填成 -1。
    
    #根据阈值，决定是否分配真实边界框
    max_ious,indices = torch.max(jaccard,dim=1)
#     对每个 anchor（沿 dim=1 在所有 gt 上取最大）：
# max_ious：形状 (num_anchors,)，每个 anchor 与所有 gt 的最大 IoU
# indices：形状 (num_anchors,)，对应最大 IoU 的 gt 索引
#指定维度会，返回在这个维度上的索引
    
    anc_i = torch.nonzero(max_ious >= iou_threshold).reshape(-1)
#     找出哪些 anchor 的最大 IoU ≥ 阈值：
# max_ious >= threshold 是一个 bool 向量 (num_anchors,)
# torch.nonzero(...) 得到满足条件的索引（形状通常 (K,1)）
# .reshape(-1) 拉平成 (K,)
# anc_i 是这些“正样本 anchor”的索引列表
    
    box_j = indices[max_ious >= iou_threshold]
    anchors_bbox_map[anc_i] = box_j
    col_discard = torch.full((num_anchors,),-1)
    row_discard = torch.full((num_gt_bboxes,),-1)
    for _ in range(num_gt_bboxes):
        max_idx = torch.argmax(jaccard)
        box_idx = (max_idx % num_gt_boxes).long()
        anc_idx = (max_idx / num_gt_boxes).long()
        anchors_bbox_map[anc_idx] = box_idx
        jaccard[:,box_idx] = col_discard
        jaccard[anc_idx,:] = row_discard
    return anchors_bbox_map

In [8]:
#标注类别和偏移量
def offset_boxes(anchors,assigned_bb,eps=1e-6):
    """对锚框偏移量的转换"""
    c_anc = d2l.box_corner_to_center(anchors)
#     把 anchor 从角点坐标转换为中心点坐标。
# 假设 anchor 角点是 (xmin, ymin, xmax, ymax)，转换后通常是：
# cx = (xmin + xmax) / 2
# cy = (ymin + ymax) / 2
# w = xmax - xmin
# h = ymax - ymin
# 所以 c_anc 的形状是 (N,4)，每行为 (cx, cy, w, h)。
    
    c_assigned_bb = d2l.box_corner_to_center(assigned_bb)
    offset_xy = 10*(c_assigned_bb[:,:2]-c_anc[:,:2])/c_anc[:,2:]
#     c_assigned_bb[:, :2] 取真实框中心 (cx_gt, cy_gt)，形状 (N,2)
# c_anc[:, :2] 取 anchor 中心 (cx_a, cy_a)，形状 (N,2)
# 差值 c_assigned_bb[:,:2] - c_anc[:,:2] 得到中心位移 (cx_gt - cx_a, cy_gt - cy_a)，形状 (N,2)
# 然后除以 c_anc[:, 2:]：
# c_anc[:, 2:] 是 anchor 的 (w_a, h_a)，形状 (N,2)
# 这一步是在做归一化
    
    offset_wh = 5*torch.log(eps + c_assigned_bb[:,2:]/c_anc[:,2:])
#     c_assigned_bb[:, 2:] 是真实框 (w_gt, h_gt)，形状 (N,2)
# c_anc[:, 2:] 是 anchor (w_a, h_a)，形状 (N,2)
# 比值 w_gt / w_a、h_gt / h_a 表达“真实框相对 anchor 放大/缩小多少”。
# 为什么取 log？
# 宽高是乘法关系（放大/缩小比例），用 log 变成加法关系，更适合回归：
# 果 w_gt = w_a，比值为 1，log(1)=0，表示“不需要缩放”。
# 为什么加 eps？
# 防止 w_gt 或 h_gt 为 0（或数值极小）导致 log(0)。
# 理论上合法框不会为 0，但加上更稳。
# 最后乘以 5：
# 同样是经验缩放因子，让回归目标尺度更合适。
# 所以 offset_wh 形状 (N,2)，内容是 [Δw, Δh]（已乘 5）。
    
    offset = torch.cat([offset_xy,offset_wh],axis=1)
#     把 (N,2) 的 offset_xy 和 (N,2) 的 offset_wh 在列方向拼起来：
# axis=1 表示沿特征维拼接
# 输出 offset 形状 (N,4)，每行是：[Δ𝑥,Δ𝑦,Δ𝑤,Δℎ ]
    
    return offset

In [9]:
#@save
def multibox_target(anchors, labels):
    """使用真实边界框标记锚框"""
    batch_size, anchors = labels.shape[0], anchors.squeeze(0)
#     batch_size = labels.shape[0]：batch 的图片数量。
# anchors = anchors.squeeze(0)：把 anchors 的第 0 维去掉。
# 如果 anchors 原来是 (1, num_anchors, 4)，squeeze 后变成 (num_anchors, 4)。
# 为什么要 squeeze：后面要对同一套 anchors 给 batch 内每张图做标注，anchors 不需要 batch 维。
    
    batch_offset, batch_mask, batch_class_labels = [], [], []
    device, num_anchors = anchors.device, anchors.shape[0]
    for i in range(batch_size):
        label = labels[i, :, :]
#         取出第 i 张图片的全部真实框：
# label 形状通常 (num_gt, 5)
# 每行 [class_id, x1, y1, x2, y2]
        anchors_bbox_map = assign_anchor_to_bbox(
            label[:, 1:], anchors, device)
        bbox_mask = ((anchors_bbox_map >= 0).float().unsqueeze(-1)).repeat(
            1, 4)
        # 将类标签和分配的边界框坐标初始化为零
        class_labels = torch.zeros(num_anchors, dtype=torch.long,
                                   device=device)
        assigned_bb = torch.zeros((num_anchors, 4), dtype=torch.float32,
                                  device=device)
        # 使用真实边界框来标记锚框的类别。
        # 如果一个锚框没有被分配，标记其为背景（值为零）
        indices_true = torch.nonzero(anchors_bbox_map >= 0)
        bb_idx = anchors_bbox_map[indices_true]
        class_labels[indices_true] = label[bb_idx, 0].long() + 1
        assigned_bb[indices_true] = label[bb_idx, 1:]
        # 偏移量转换
        offset = offset_boxes(anchors, assigned_bb) * bbox_mask
        batch_offset.append(offset.reshape(-1))
        batch_mask.append(bbox_mask.reshape(-1))
        batch_class_labels.append(class_labels)
    bbox_offset = torch.stack(batch_offset)
    bbox_mask = torch.stack(batch_mask)
    class_labels = torch.stack(batch_class_labels)
    return (bbox_offset, bbox_mask, class_labels)

In [10]:
ground_truth = torch.tensor([[0, 0.1, 0.08, 0.52, 0.92],
                         [1, 0.55, 0.2, 0.9, 0.88]])
anchors = torch.tensor([[0, 0.1, 0.2, 0.3], [0.15, 0.2, 0.4, 0.4],
                    [0.63, 0.05, 0.88, 0.98], [0.66, 0.45, 0.8, 0.8],
                    [0.57, 0.3, 0.92, 0.9]])

fig = d2l.plt.imshow(img)
show_bboxes(fig.axes, ground_truth[:, 1:] * bbox_scale, ['dog', 'cat'], 'k')
show_bboxes(fig.axes, anchors * bbox_scale, ['0', '1', '2', '3', '4']);

NameError: name 'img' is not defined